# 41 — Unsupervised Learning

## Objective

Understand the purpose of unsupervised learning and how it differs from supervised learning.

In supervised learning:

- We have input features `X`
- We have a known target `y`
- The model learns a relationship between `X` and `y`

In unsupervised learning:

- We have input features `X`
- There is no target `y`
- The algorithm tries to discover useful structure in the data

Main areas:

1. Clustering
2. Dimensionality reduction
3. Representation learning

This notebook focuses on the fundamentals.

K-Means clustering will be covered separately.

## 1. Supervised vs Unsupervised Learning

### Supervised Learning

The dataset contains examples of:

`features → known target`

Example:

| sqft | bedrooms | price |
|------|----------|-------|
| 1000 | 2 | ₹50L |
| 1500 | 3 | ₹80L |

The model learns to predict the target.

Examples:

- Linear Regression
- Logistic Regression
- Decision Trees
- Random Forest
- SVM
- KNN

### Unsupervised Learning

The dataset contains features but no known target.

Example:

| sqft | bedrooms | location |
|------|----------|----------|
| 1000 | 2 | A |
| 1500 | 3 | B |
| 900 | 2 | A |

The goal may be to discover groups or other structure in the data.

Examples:

- Customer segmentation
- Property segmentation
- Anomaly detection
- Dimensionality reduction

In [4]:
import pandas as pd
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)

df = wine.frame

df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [2]:
df.shape

(178, 14)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  target          

## 2. Important Observation

The Wine dataset contains a `target` column because the original dataset is labeled.

However, for this unsupervised-learning exercise, we will **not use the target**.

Our model will only see the feature columns.

This is an important distinction:

The dataset may contain labels, but an unsupervised algorithm does not need those labels to discover structure.

In [6]:
X = df.drop(columns="target")

X.shape

(178, 13)

## 3. What Is Clustering?

Clustering attempts to divide observations into groups based on similarity.

For example, suppose we have thousands of customers.

We may not know their customer segment beforehand.

A clustering algorithm could potentially discover groups such as:

- customers with low spending
- customers with medium spending
- customers with high spending

The important point is:

The algorithm does not receive these group names beforehand.

It discovers groups from the feature values.

### Classification vs Clustering

Classification:

`features → known class`

Clustering:

`features → discovered groups`

Therefore, clustering is not simply "classification without labels".

The objective and interpretation are different.

## 4. Why Clustering Is Useful

Common applications include:

### Customer segmentation

Group customers according to behavior.

### Real estate

Group properties according to characteristics such as:

- area
- price
- bedrooms
- location-related features

### Anomaly detection

Identify observations that do not fit normal patterns.

### Document analysis

Group similar documents or text representations.

### Image processing

Group pixels or image representations according to similarity.

The usefulness of clustering depends heavily on whether the discovered groups have meaningful interpretation.

In [7]:
X.describe()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
count,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000,178.000000
mean,13.000618,2.336348,2.366517,19.494944,99.741573,2.295112,2.029270,0.361854,1.590899,5.058090,0.957449,2.611685,746.893258
std,0.811827,1.117146,0.274344,3.339564,14.282484,0.625851,0.998859,0.124453,0.572359,2.318286,0.228572,0.709990,314.907474
min,11.030000,0.740000,1.360000,10.600000,70.000000,0.980000,0.340000,0.130000,0.410000,1.280000,0.480000,1.270000,278.000000
25%,12.362500,1.602500,2.210000,17.200000,88.000000,1.742500,1.205000,0.270000,1.250000,3.220000,0.782500,1.937500,500.500000
50%,13.050000,1.865000,2.360000,19.500000,98.000000,2.355000,2.135000,0.340000,1.555000,4.690000,0.965000,2.780000,673.500000
75%,13.677500,3.082500,2.557500,21.500000,107.000000,2.800000,2.875000,0.437500,1.950000,6.200000,1.120000,3.170000,985.000000
max,14.830000,5.800000,3.230000,30.000000,162.000000,3.880000,5.080000,0.660000,3.580000,13.000000,1.710000,4.000000,1680.000000


## 6. Why Feature Scale Matters

Many clustering algorithms use distances between observations.

Consider two features:

- income: 20,000 – 200,000
- age: 18 – 80

The income feature has a much larger numerical scale.

Without appropriate preprocessing, distance-based algorithms can give disproportionate influence to features with larger scales.

Therefore, preprocessing such as feature scaling can be important before distance-based clustering.

This is one reason why the preprocessing concepts learned earlier are also important for unsupervised learning.

## 7. No Target Does Not Mean No Evaluation

A common misunderstanding is:

> "Because there is no target, we cannot evaluate an unsupervised model."

We can evaluate clustering, but evaluation is different from supervised learning.

Possible approaches include:

- Silhouette score
- Inertia for K-Means
- Cluster stability
- Domain/business interpretation
- External labels, when available, used only for analysis

There is usually no single metric that proves that a clustering solution is "correct".

A mathematically good cluster may still be useless for the actual business problem.

## 8. Important Warning

Unsupervised learning discovers structure.

It does not automatically discover meaningful truth.

For example:

A clustering algorithm may divide customers into three groups.

That does not automatically mean that the business has exactly three real customer segments.

The number of clusters, selected features, preprocessing, algorithm, and interpretation can all affect the result.

Therefore:

**Do not over-interpret clustering results.**

## 9. Industry Workflow

A practical unsupervised-learning workflow is:

1. Define the business/problem objective
2. Collect and understand the data
3. Select useful features
4. Clean the data
5. Handle missing values
6. Scale/transform features when appropriate
7. Apply an unsupervised algorithm
8. Evaluate the discovered structure
9. Profile the resulting groups
10. Validate whether the groups are useful in the real problem

The algorithm is only one part of the workflow.

## 10. What We Will Learn Next

The next notebook will focus specifically on:

# K-Means Clustering

We will learn:

- How K-Means works
- Centroids
- Distance-based assignment
- Iterative updates
- `n_clusters`
- `random_state`
- `n_init`
- Inertia
- Choosing K
- Elbow method
- Silhouette score
- Feature scaling
- Interpreting clusters
- Applying K-Means to real data

We will use a real dataset rather than a manually-created toy dataset.

## Key Takeaways

- Unsupervised learning works without a target variable.
- Clustering discovers groups based on feature similarity.
- Classification predicts known classes; clustering discovers groups.
- Feature scaling can matter for distance-based algorithms.
- Unsupervised results require interpretation.
- There is no guarantee that discovered clusters represent real-world categories.
- K-Means is one important clustering algorithm, but not the only one.
- The next notebook focuses on K-Means.